# Assignment 7 — GRPO Fine-Tuning



In [1]:
# ============================================================
# 1. Install/upgrade packages for a 2026 Colab runtime
# ============================================================
# Recommended runtime: GPU
# In Colab: Runtime -> Change runtime type -> T4 GPU or better

import os
import sys
import subprocess
import locale

# Fix common Colab/Windows locale issue used by some libraries
locale.getpreferredencoding = lambda: "UTF-8"

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


core_packages = [
    "transformers>=4.56.0",
    "trl>=0.29.0",
    "peft>=0.17.0",
    "accelerate>=1.10.0",
    "datasets>=4.0.0",
    "bitsandbytes>=0.46.0",
    "sentencepiece",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *core_packages])

# Optional: install Unsloth. If it fails, the notebook will still work using vanilla TRL + PEFT.
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "unsloth", "unsloth_zoo"])
    print("Unsloth installed successfully.")
except Exception as e:
    print("Unsloth installation failed. The notebook will use vanilla TRL + PEFT.")
    print("Reason:", e)

print("Install cell complete. If Colab asks you to restart the runtime, restart and run again from the imports cell.")

Unsloth installed successfully.
Install cell complete. If Colab asks you to restart the runtime, restart and run again from the imports cell.


In [2]:
# ============================================================
# 2. Imports and environment check
# ============================================================

import os
import re
import math
import torch

# Import Unsloth first if available, before importing TRL/Transformers trainers.
try:
    from unsloth import FastLanguageModel, PatchFastRL
    from unsloth import is_bfloat16_supported
    PatchFastRL("GRPO", FastLanguageModel)
    HAS_UNSLOTH = True
except Exception as e:
    HAS_UNSLOTH = False
    print("Unsloth is not available. Falling back to vanilla TRL + PEFT.")
    print("Reason:", e)

    def is_bfloat16_supported():
        return torch.cuda.is_available() and torch.cuda.is_bf16_supported()


from datasets import load_dataset
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", is_bfloat16_supported())
else:
    print("Warning: GRPO fine-tuning is very slow without a GPU.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.
Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
bf16 supported: False


In [3]:
# ============================================================
# 3. Model configuration
# ============================================================


MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

MAX_SEQ_LENGTH = 1024
LORA_RANK = 16

# vLLM is optional. Keep False for maximum Colab compatibility.
# If you have a strong GPU and install vllm manually, you can set this to True.
USE_VLLM = False

# Prefer Unsloth if it imported correctly; otherwise use standard Transformers + PEFT.
USE_UNSLOTH = HAS_UNSLOTH

print("MODEL_NAME:", MODEL_NAME)
print("USE_UNSLOTH:", USE_UNSLOTH)
print("USE_VLLM:", USE_VLLM)


MODEL_NAME: Qwen/Qwen2.5-0.5B-Instruct
USE_UNSLOTH: True
USE_VLLM: False


In [4]:
# ============================================================
# 4. Load model and tokenizer
# ============================================================

if USE_UNSLOTH:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        fast_inference=USE_VLLM,
        max_lora_rank=LORA_RANK,
        gpu_memory_utilization=0.45,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=LORA_RANK,
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        bias="none",
    )
    peft_config = None

else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import get_peft_model, prepare_model_for_kbit_training

    compute_dtype = torch.bfloat16 if is_bfloat16_supported() else torch.float16

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        torch_dtype=compute_dtype,
        device_map="auto",
        trust_remote_code=True,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)

    peft_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_RANK,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=0.0,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_config)
    peft_config = None

print("Model and tokenizer loaded.")

==((====))==  Unsloth 2026.5.10: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

... (4 outputs left out)

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.10 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


Model and tokenizer loaded.


## Dataset preparation

We use GSM8K for math reasoning. The model receives a prompt and learns to produce a structured answer.  
The answer format is rewarded using custom reward functions.

In [5]:
# ============================================================
# 5. Load GSM8K dataset
# ============================================================

raw_dataset = load_dataset("openai/gsm8k", "main", split="train[:256]")
raw_dataset


README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answer'],
    num_rows: 256
})

In [6]:
# ============================================================
# 6. Format the dataset for GRPO
# ============================================================

SYSTEM_PROMPT = '''Please solve the math word problem using the exact format below.

Format:
<reasoning>
Write a clear but concise solution strategy and calculations.
</reasoning>

Step-by-step explanation:
1) ...
2) ...
Answer = <number> <unit>

Important:
- Always include <reasoning> and </reasoning>.
- Always end with Answer = <number> <unit>.
- Put only the final numeric value after Answer =.
'''

def extract_numeric_answer_from_gsm8k(text):
    '''Extract the GSM8K final answer after ####.'''
    match = re.search(r"####\s*(-?[\d,]+(?:\.\d+)?)", text)
    if not match:
        return None
    return match.group(1).replace(",", "").strip()

def format_dataset(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["question"]},
        ],
        "answer": extract_numeric_answer_from_gsm8k(example["answer"]),
    }

dataset = raw_dataset.map(format_dataset)
dataset = dataset.remove_columns([col for col in raw_dataset.column_names if col in dataset.column_names and col not in ["prompt", "answer"]])

print(dataset)
dataset[0]

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

Dataset({
    features: ['answer', 'prompt'],
    num_rows: 256
})


{'answer': '72',
 'prompt': [{'role': 'system',
   'content': 'Please solve the math word problem using the exact format below.\n \nFormat:\n<reasoning>\nWrite a clear but concise solution strategy and calculations.\n</reasoning>\n \nStep-by-step explanation:\n1) ...\n2) ...\nAnswer = <number> <unit>\n \nImportant:\n- Always include <reasoning> and </reasoning>.\n- Always end with Answer = <number> <unit>.\n- Put only the final numeric value after Answer =.\n'},
  {'role': 'user',
   'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'}]}

## Reward functions

GRPO uses reward functions to score the model completions.  
The functions below reward:
1. Correct final numeric answer.
2. Correct output format.
3. Reasoning length within a useful range.
4. Explanation length within a useful range.

In [7]:
# ============================================================
# 7. Helper functions for completions and answers
# ============================================================

def completion_to_text(completion):
    """Handle both string completions and chat-style completion objects."""
    if isinstance(completion, str):
        return completion

    if isinstance(completion, dict):
        return completion.get("content", str(completion))

    if isinstance(completion, list):
        if len(completion) == 0:
            return ""
        # TRL chat-style completion often looks like:
        # [{"role": "assistant", "content": "..."}]
        if isinstance(completion[-1], dict):
            return completion[-1].get("content", "")
        return str(completion[-1])

    return str(completion)


def extract_answer_from_model_output(text):
    """Extract numeric value after 'Answer =' from model output."""
    match = re.search(r"Answer\s*=\s*(-?[\d,]+(?:\.\d+)?)", text, flags=re.IGNORECASE)
    if not match:
        return None
    return match.group(1).replace(",", "").strip()


def same_numeric_answer(predicted, gold, tolerance=1e-6):
    """Compare numeric answers robustly."""
    if predicted is None or gold is None:
        return False
    try:
        return abs(float(predicted) - float(gold)) <= tolerance
    except Exception:
        return str(predicted).strip() == str(gold).strip()

In [8]:
# ============================================================
# 8. Reward 1: answer correctness
# ============================================================

def answer_reward(completions, answer, **kwargs):
    rewards = []

    for completion, gold_answer in zip(completions, answer):
        response = completion_to_text(completion)
        predicted_answer = extract_answer_from_model_output(response)

        if same_numeric_answer(predicted_answer, gold_answer):
            rewards.append(1.0)
        elif predicted_answer is not None:
            rewards.append(0.2)
        else:
            rewards.append(0.0)

    return rewards

In [9]:
# ============================================================
# 9. Reward 2: required format
# ============================================================

def format_reward(completions, **kwargs):
    pattern = re.compile(
        r"^\s*<reasoning>[\s\S]+?</reasoning>[\s\S]+?Answer\s*=\s*-?[\d,]+(?:\.\d+)?",
        flags=re.IGNORECASE,
    )

    responses = [completion_to_text(completion) for completion in completions]
    return [0.5 if pattern.search(response) else 0.0 for response in responses]

In [10]:
# ============================================================
# 10. Reward 3: reasoning length
# ============================================================

def reasoning_length_reward(completions, **kwargs):
    pattern = re.compile(r"<reasoning>([\s\S]+?)</reasoning>", flags=re.IGNORECASE)
    responses = [completion_to_text(completion) for completion in completions]
    rewards = []

    for response in responses:
        match = pattern.search(response)
        if not match:
            rewards.append(0.0)
            continue

        reasoning_text = match.group(1).strip()
        reasoning_length = len(reasoning_text)

        if 120 <= reasoning_length <= 1000:
            rewards.append(0.5)
        else:
            rewards.append(0.0)

    return rewards

In [11]:
# ============================================================
# 11. Reward 4: explanation length
# ============================================================

def explanation_length_reward(completions, **kwargs):
    pattern = re.compile(r"</reasoning>([\s\S]+)", flags=re.IGNORECASE)
    responses = [completion_to_text(completion) for completion in completions]
    rewards = []

    for response in responses:
        match = pattern.search(response)
        if not match:
            rewards.append(0.0)
            continue

        explanation_text = match.group(1).strip()
        explanation_length = len(explanation_text)

        if 80 <= explanation_length <= 800:
            rewards.append(0.5)
        else:
            rewards.append(0.0)

    return rewards

In [12]:
# ============================================================
# 12. Quick reward-function test
# ============================================================

test_completions = [
    [{"role": "assistant", "content": """<reasoning>
We need half of 48, which is 24. Then we add April and May: 48 + 24 = 72.
</reasoning>

Step-by-step explanation:
1) April sales = 48 clips.
2) May sales = 48 / 2 = 24 clips.
3) Total = 48 + 24 = 72 clips.
Answer = 72 clips."""}],
    [{"role": "assistant", "content": "The answer is probably 70."}],
]

test_answers = ["72", "72"]

print("answer_reward:", answer_reward(test_completions, test_answers))
print("format_reward:", format_reward(test_completions))
print("reasoning_length_reward:", reasoning_length_reward(test_completions))
print("explanation_length_reward:", explanation_length_reward(test_completions))

answer_reward: [1.0, 0.0]
format_reward: [0.5, 0.0]
reasoning_length_reward: [0.0, 0.0]
explanation_length_reward: [0.5, 0.0]


## GRPO training configuration



In [13]:
# ============================================================
# 13. GRPO training arguments
# ============================================================

training_args = GRPOConfig(
    output_dir="./results_grpo_2026",

    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    num_generations=2,

    max_prompt_length=256,
    max_completion_length=128,

    # Training hyperparameters
    learning_rate=5e-5,
    max_steps=20,
    save_steps=20,
    logging_steps=1,
    warmup_steps=2,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,

    # Mixed precision
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),

    # Keep vLLM disabled by default for Colab reliability.
    use_vllm=USE_VLLM,

    # 8-bit optimizer is memory efficient on GPU.
    optim="paged_adamw_8bit",

    report_to="none",
    remove_unused_columns=False,
)

training_args


UnslothGRPOConfig(output_dir='./results_grpo_2026', per_device_train_batch_size=2, num_train_epochs=3.0, max_steps=20, learning_rate=5e-05, lr_scheduler_type=<SchedulerType.COSINE: 'cosine'>, lr_scheduler_kwargs=None, warmup_steps=2, optim=<OptimizerNames.PAGED_ADAMW_8BIT: 'paged_adamw_8bit'>, optim_args=None, weight_decay=0.01, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, optim_target_modules=None, gradient_accumulation_steps=1, average_tokens_across_devices=True, max_grad_norm=0.3, label_smoothing_factor=0.0, bf16=False, fp16=True, bf16_full_eval=False, fp16_full_eval=False, tf32=None, gradient_checkpointing=True, gradient_checkpointing_kwargs=None, torch_compile=False, torch_compile_backend=None, torch_compile_mode=None, use_liger_kernel=False, liger_kernel_config=None, use_cache=False, neftune_noise_alpha=None, torch_empty_cache_steps=250, auto_find_batch_size=False, logging_strategy=<IntervalStrategy.STEPS: 'steps'>, logging_steps=1, logging_first_step=False, log_on_each_

In [14]:
# ============================================================
# 14. Create GRPO trainer
# ============================================================

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        answer_reward,
        format_reward,
        reasoning_length_reward,
        explanation_length_reward,
    ],
    args=training_args,
    train_dataset=dataset,
)

print("Trainer created successfully.")

Trainer created successfully.


In [15]:
# ============================================================
# 15. Start GRPO training
# ============================================================

trainer.train()




The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 256 | Num Epochs = 1 | Total steps = 20
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'disable_compile', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will ta

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / answer_reward / mean,rewards / answer_reward / std,rewards / format_reward / mean,rewards / format_reward / std,rewards / reasoning_length_reward / mean,rewards / reasoning_length_reward / std,rewards / explanation_length_reward / mean,rewards / explanation_length_reward / std
1,-0.000000,0.000000,0.000000,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,-0.000000,0.000000,0.000000,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,104.000000,80.000000,128.000000,0.500000,80.000000,80.000000,80.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,-0.000000,0.000000,0.000000,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,0.000000,0.000000,0.000000,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,0.000000,0.000000,0.000000,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,-0.000000,0.000000,0.000000,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,-0.000000,0.000000,0.000000,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
10,-0.000000,0.000000,0.000000,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1

TrainOutput(global_step=20, training_loss=1.5223480461284066e-08, metrics={'train_runtime': 244.6847, 'train_samples_per_second': 0.163, 'train_steps_per_second': 0.082, 'total_flos': 0.0, 'train_loss': 1.5223480461284066e-08})